# Multimodal Classification
## Loading & Training Unimodal Classifiers
Documentation can be found in Jupyter files 'Biomarker Classification', 'Genetic Classification', 'Clinical Classification', and 'Multimodal Dataset Preprocessing'.

### Clinical X Datasets

In [ ]:
# loading train, test, and validation datasets
X_train = pd.read_pickle(r"C:\Users\kishe\Documents\Year 3 Jupyter\X_train_merged.pkl")
X_val = pd.read_pickle(r"C:\Users\kishe\Documents\Year 3 Jupyter\X_val_merged.pkl")
X_test = pd.read_pickle(r"C:\Users\kishe\Documents\Year 3 Jupyter\X_test_merged.pkl")

print(type(X_train))
print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

In [ ]:
# columns to add to the X_val dataset which are missing
columns_to_add = ['PTETHCAT_Unknown', 'PTRACCAT_Asian', 'PTRACCAT_More than one', 'PTMARRY_Unknown', 'PTETHCAT_Not Hisp/Latino']

# adding new columns filled with zeros
for col in columns_to_add:
    X_val[col] = 0

print(X_val.head())

In [ ]:
# columns to add to the X_test dataset which are missing
columns_to_add = ['PTETHCAT_Unknown', 'PTRACCAT_Asian', 'PTMARRY_Unknown', 'PTETHCAT_Not Hisp/Latino']

# adding new columns filled with zeros
for col in columns_to_add:
    X_test[col] = 0

print(X_test.head())

In [ ]:
# extracting clinical data from the merged dataset and creating clinical train, val and test datasets
clinical_columns = ['AGE', 'PTEDUCAT', 'FDG', 'ABETA', 'TAU', 'PTAU', 'CDRSB', 'ADAS11', 'ADAS13', 'ADASQ4', 'MMSE', 
                'RAVLT_immediate', 'RAVLT_learning', 'RAVLT_forgetting', 'RAVLT_perc_forgetting', 'LDELTOTAL',
                'DIGITSCOR', 'TRABSCOR', 'FAQ', 'Ventricles', 'Hippocampus', 'WholeBrain', 'Entorhinal',
                'Fusiform', 'MidTemp', 'ICV', 'mPACCdigit', 'mPACCtrailsB', 'PTGENDER_Male', 'PTETHCAT_Not Hisp/Latino',
                'PTETHCAT_Unknown', 'PTRACCAT_Black', 
                'PTRACCAT_White', 'PTMARRY_Married','PTMARRY_Widowed', 'APOE4_1.0', 'APOE4_2.0'
                # 'PTRACCAT_Asian', 'PTMARRY_Never married', 'PTMARRY_Unknown', 'PTRACCAT_More than one'
                ]
X_train_clinical = X_train[clinical_columns]
X_val_clinical = X_val[clinical_columns]
X_test_clinical = X_test[clinical_columns]

print(X_train_clinical.head(10))

### Genetic X Datasets

In [ ]:
# extracting the genetic features from the merged datasets
genetic_columns = ['ABCA7', 'ADAM10', 'APOE', 'APP', 'CD2AP', 'CLU', 'LRRK2',
       'PICALM', 'SORL1', 'TREM2']
X_train_genetic = X_train[genetic_columns]
X_val_genetic = X_val[genetic_columns]
X_test_genetic = X_test[genetic_columns]

print(X_train_genetic.shape)
print(X_val_genetic.shape)
print(X_test_genetic.shape)
print(X_train_genetic.head(10))

### Biomarker X Datasets

In [ ]:
biomarker_columns = ['CTWHITE', 'CTRED', 'PROTEIN', 'GLUCOSE', 'CTWHITE_M12',
       'CTRED_M12', 'PROTEIN_M12', 'GLUCOSE_M12', 'CTWHITE_M24', 'CTRED_M24',
       'PROTEIN_M24', 'GLUCOSE_M24', 'CTWHITE_M36', 'CTRED_M36', 'PROTEIN_M36',
       'GLUCOSE_M36', 'CTWHITE_M48', 'CTRED_M48', 'PROTEIN_M48',
       'GLUCOSE_M48']
X_train_biomarker = X_train[biomarker_columns]
X_val_biomarker = X_val[biomarker_columns]
X_test_biomarker = X_test[biomarker_columns]

In [ ]:
# converting DataFrame to numpy array
X_train_array = X_train_biomarker.values
X_val_array = X_val_biomarker.values
X_test_array = X_test_biomarker.values

# reshaping X_train, X_val and X_test
X_train_rnn = X_train_array.reshape(-1, 5, 4)
print("Shape of X_train_rnn:", X_train_rnn.shape)

X_val_rnn = X_val_array.reshape(-1, 5, 4)
print("Shape of X_val_rnn:", X_val_rnn.shape)

X_test_rnn = X_test_array.reshape(-1, 5, 4)
print("Shape of X_test_rnn:", X_test_rnn.shape)

### Diagnosis Y Labels

In [ ]:
# extracting the diagnosis labels from the merged dataset
y_train = X_train["Diagnosis"]
y_val = X_val["Diagnosis"]
y_test = X_test["Diagnosis"]

# flattening to 1D array
y_train = y_train.values.ravel()
y_val = y_val.values.ravel() 
y_test = y_test.values.ravel()

print(type(y_train))
print(y_train[:20])

In [ ]:
# encoding the categorical diagnosis labels with numerical mappings
label_mapping = {"AD": 0, "CN": 1, "MCI": 2}
y_train = np.array([label_mapping[label] for label in y_train])
y_val = np.array([label_mapping[label] for label in y_val])
y_test = np.array([label_mapping[label] for label in y_test])

print(type(y_train))
print(y_train[:20])

### Clinical Classifier
Code evaluation loop developed with guidance and adapted from: https://github.com/rsinghlab/MADDi/blob/main/training/train_clinical.py

In [ ]:
# setting random seeds for reproducibility
def reset_random_seeds(seed):
    os.environ['PYTHONHASHSEED']=str(seed)  
    tf.random.set_seed(seed)  
    np.random.seed(seed)  
    random.seed(seed)  

# lists to store evaluation metrics
acc = []
f1 = []
precision = []
recall = []

# generating random seeds for experiments
seeds = seeds = [42, 10, 53, 78, 20]

# looping through each seed
for seed in seeds:
    reset_random_seeds(seed)
    print("Seed:", seed)
    # defining the neural network model and adding layers
    model = Sequential()
    model.add(Dense(128, input_shape = (37,), activation = "relu"))
    model.add(BatchNormalization())
    model.add(Dropout(0.5))
    model.add(Dense(64, activation = "relu"))
    model.add(BatchNormalization())
    model.add(Dropout(0.3))
    model.add(Dense(50, activation = "relu"))
    model.add(BatchNormalization())
    model.add(Dropout(0.2))
    model.add(Dense(3, activation = "softmax"))

    # compiling the model
    model.compile(Adam(learning_rate = 0.001), "sparse_categorical_crossentropy", metrics = ["sparse_categorical_accuracy"])

    model.summary()

    # training and evaluating the model
    history = model.fit(X_train_clinical, y_train, epochs=250,
                        validation_data=(X_val_clinical, y_val),
                        batch_size=16, verbose=1)
    score = model.evaluate(X_test_clinical, y_test, verbose=0)
    print(f'Test loss: {score[0]} / Test accuracy: {score[1]}')
    acc.append(score[1])

    # making predictions on test data
    y_pred = model.predict(X_test_clinical)
    y_pred = np.argmax(y_pred, axis=1)

    # calculating classification report
    cr = classification_report(y_test, y_pred, output_dict=True)
    precision.append(cr["macro avg"]["precision"])
    recall.append(cr["macro avg"]["recall"])
    f1.append(cr["macro avg"]["f1-score"])

print("Avg accuracy: " + str(np.array(acc).mean()))
print("Avg precision: " + str(np.array(precision).mean()))
print("Avg recall: " + str(np.array(recall).mean()))
print("Avg f1: " + str(np.array(f1).mean()))
print("Std accuracy: " + str(np.array(acc).std()))
print("Std precision: " + str(np.array(precision).std()))
print("Std recall: " + str(np.array(recall).std()))
print("Std f1: " + str(np.array(f1).std()))
print(acc)
print(precision)
print(recall)
print(f1)

In [ ]:
clinical_predictions = y_pred

### Genetic Classifier

In [ ]:
# creating the gradient boosting model
gb_model = GradientBoostingClassifier(n_estimators=1000, 
                                      learning_rate=0.001, 
                                      max_depth=500, 
                                      min_samples_split=115, 
                                      min_samples_leaf=55, 
                                      )

# fitting the model on training data and predicting labels for test data
gb_model.fit(X_train_genetic, y_train)
y_pred = gb_model.predict(X_test_genetic)

# evaluating the model on validation and test data
validation_accuracy = gb_model.score(X_val_genetic, y_val)
print("Validation Accuracy:", validation_accuracy)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='macro')
recall = recall_score(y_test, y_pred, average='macro')
f1 = f1_score(y_test, y_pred, average='macro')

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

# printing a confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.xlabel('Predicted labels')
plt.ylabel('True labels')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
genetic_predictions = y_pred

### Biomarker Classifier

In [ ]:
model = Sequential([
    LSTM(units=200, input_shape=(5, 4), return_sequences=True),  
    LSTM(units=100, return_sequences=True), 
    LSTM(units=50),  
    Dense(16, activation='relu'), 
    Dense(3, activation='softmax')  
])

# compiling model
model.compile(optimizer=Adam(0.001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# training the model
model.fit(X_train_rnn, y_train, epochs=64, batch_size=16, validation_data=(X_val_rnn, y_val))

# predicting probabilities for each class
y_probs = model.predict(X_test_rnn)
y_pred = np.argmax(y_probs, axis=1)

# calculating confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(conf_matrix)

# calculating precision, recall, f1-score, and accuracy
report = classification_report(y_test, y_pred, target_names=['AD', 'CN', 'MCI'])
print("Classification Report:")
print(report)
accuracy = np.sum(y_test == y_pred) / len(y_test)
print("Accuracy:", accuracy)

plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.xlabel('Predicted labels')
plt.ylabel('True labels')
plt.title(f'Confusion Matrix')
plt.show()

In [ ]:
biomarker_predictions = y_pred

## Late Fusion

Late fusion classification combines predictions from multiple classifiers, each trained on different modalities or features of the data, to make a final prediction. This code will concatenate the predictions at each index from the biomarker, genetic, and clinical prediction sets and then train a further classifier to generate a final overall prediction.

In the first method, concatenation, predictions from unimodal classifiers for each instance are simply concatenated together into a single feature vector. This assumes that each modality contributes equally to the final prediction.

The second method, weighted concatenation, assigns different weights to the predictions from each modality before concatenating them. In this case, the predictions from the clinical classifier are given a weight of 0.5, the genetic classifier predictions a weight of 0.4, and the biomarker classifier predictions a weight of 0.3. These weights reflect the relative importance of each modality in making the final prediction.

Concatenating the predictions from unimodal classifiers works to make a more accurate multimodal prediction because it leverages the complementary information present in different modalities. Each modality may capture different aspects or features of the data. Additionally, by assigning weights to the predictions in the weighted concatenation method, the importance of certain modalities over others can be highlighted with their relative contributions to the final prediction.

In [ ]:
concat_predictions = []
for i in range(len(clinical_predictions)):  
    concat_prediction = [clinical_predictions[i], genetic_predictions[i], biomarker_predictions[i]]
    concat_predictions.append(concat_prediction)

In [ ]:
weighted_concat_predictions = []
for i in range(len(clinical_predictions)):  
    weighted_concat_prediction = [
        clinical_predictions[i] * 0.4,
        genetic_predictions[i] * 0.4,
        biomarker_predictions[i] * 0.2
    ]
    weighted_concat_predictions.append(weighted_concat_prediction)

### Classifiers - Final Prediction

A late fusion classifier is implemented using concatenated predictions from multiple classifiers and a logistic regression/random forest meta-model for classification.

It first splits the combined feature vectors and true class labels into training and testing sets, with a 50% split for testing. Then, a logistic regression/random forest model (`meta_model`) is chosen as the meta-model to combine predictions from individual classifiers, trained on the concatenated feature vectors (`concat_predictions`). 

After training, the meta-model is evaluated on the testing data (`X_test_fusion`) to generate predictions (`y_pred`), and accuracy is computed using `accuracy_score`. Additionally, a confusion matrix visualizes the classifier's performance, while precision, recall, F1-score, and accuracy metrics are computed and printed out using `classification_report`. 

In [ ]:
# splitting combined feature vectors and true class labels into train and test sets
X_train_fusion, X_test_fusion, y_train_fusion, y_test_fusion = train_test_split(weighted_concat_predictions, y_test, test_size=0.5)

print(len(y_train_fusion))
print(len(y_test_fusion))

meta_model = LogisticRegression()

# training and evaluating the meta-model
meta_model.fit(X_train_fusion, y_train_fusion)

y_pred = meta_model.predict(X_test_fusion)
accuracy = accuracy_score(y_test_fusion, y_pred)
print("Accuracy:", accuracy)

# calculating confusion matrix
conf_matrix = confusion_matrix(y_test_fusion, y_pred)
print("Confusion Matrix:")
print(conf_matrix)

# calculating precision, recall, f1-score, and accuracy
report = classification_report(y_test_fusion, y_pred, target_names=['AD', 'CN', 'MCI'])
print("Classification Report:")
print(report)

# extracting precision, recall, and f1-score from the classification report
precision = report.split('\n')[2].split()[1]
recall = report.split('\n')[2].split()[2]
f1_score = report.split('\n')[2].split()[3]

print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1_score)

plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', cbar=False)

# Update x-axis labels
plt.xticks(ticks=np.arange(len(label_mapping)), labels=label_mapping.keys())

# Update y-axis labels
plt.yticks(ticks=np.arange(len(label_mapping)), labels=label_mapping.keys())

plt.xlabel('Predicted labels')
plt.ylabel('True labels')
plt.title(f'Confusion Matrix')
plt.show()


In [ ]:
import yellowbrick
from yellowbrick.classifier import ClassificationReport

visualizer = ClassificationReport(meta_model)

visualizer.fit(X_train_fusion, y_train_fusion)
visualizer.score(X_test_fusion, y_test_fusion)
visualizer.show()

In [ ]:
# splitting combined feature vectors and true class labels into train and test sets
X_train_fusion, X_test_fusion, y_train_fusion, y_test_fusion = train_test_split(weighted_concat_predictions, y_test, test_size=0.3, random_state=42)
meta_model = RandomForestClassifier(n_estimators=100, random_state=42)

# training and evaluating the meta-model
meta_model.fit(X_train_fusion, y_train_fusion)


y_pred = meta_model.predict(X_test_fusion)
accuracy = accuracy_score(y_test_fusion, y_pred)
print("Accuracy:", accuracy)

# calculating confusion matrix
conf_matrix = confusion_matrix(y_test_fusion, y_pred)
print("Confusion Matrix:")
print(conf_matrix)

# calculating precision, recall, f1-score, and accuracy
report = classification_report(y_test_fusion, y_pred, target_names=['AD', 'CN', 'MCI'])
print("Classification Report:")
print(report)

# extracting precision, recall, and f1-score from the classification report
precision = report.split('\n')[2].split()[1]
recall = report.split('\n')[2].split()[2]
f1_score = report.split('\n')[2].split()[3]

print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1_score)

plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.xlabel('Predicted labels')
plt.ylabel('True labels')
plt.title(f'Confusion Matrix')
plt.show()


## Results Discussion

The results from both the Logistic regression and Random forest classifiers demonstrate high precision, recall, and F1-score values across all classes indicating robust performance. 

The high accuracy values (96%) further confirm the effectiveness of both classifiers in accurately predicting the disease status of patients. These results are clinically significant as they suggest that the classifiers can reliably identify individuals with AD, CN, or MCI based on the features extracted from the combined data sources (genetic, clinical, and biomarker data). 

Accurate classification is crucial in clinical practice for early diagnosis, intervention planning, and monitoring disease progression, ultimately leading to timely and personalised patient care. The consistency in performance between the meta-models shows the robustness of the late fusion approach in combining predictions from multiple classifiers to achieve accurate and clinically meaningful results.